# CopyWrite AI - run on a free GPU (Google Colab or Kaggle)

**Google Colab**
1. **Runtime > Change runtime type > T4 GPU > Save**.
2. **Runtime > Run all** (or `Ctrl+F9`).
3. After about 3-5 minutes, step 4 prints **"CopyWrite AI is ready. Open: ... trycloudflare.com"**. Click that link.

If Colab says it **cannot connect to a GPU** (the free GPU quota is used up for the day), use Kaggle instead:

**Kaggle** (30 free GPU hours per week)
1. kaggle.com > Create > New Notebook > File > **Import Notebook** > GitHub, and paste
   `https://github.com/NishaadG/CopyWrite-AI/blob/main/notebooks/run_on_colab.ipynb`
2. Right panel > Session options: **Accelerator: GPU T4 x2**, **Internet: on** (needs a verified phone number once).
3. **Run all**, then click the link printed by step 4.

**Using the app:** pick a sample handwriting (or upload yours), paste your notes, click **Write my notes**.
The first click can take 1-2 extra minutes while the models finish loading; a page takes roughly 1-3 minutes.

**If anything fails:** run step 5 and copy its output. Tick *More options > Save debug images* in the app to get every
intermediate image as a zip.

To get the newest code after a `git push`: re-run steps 2 and 4 (step 2 pulls the latest version).

In [ ]:
# 1. Check the GPU
import subprocess
try:
    print(subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"]).decode())
except Exception:
    print("NO GPU FOUND. Colab: Runtime > Change runtime type > T4 GPU. Kaggle: Accelerator > GPU T4 x2.")
    print("Without a GPU the app still runs, but with a plain font instead of the AI model.")

In [ ]:
# 2. Get the code (clones the first time, pulls updates afterwards)
import os
REPO_URL = "https://github.com/NishaadG/CopyWrite-AI.git"
BASE = next(p for p in ["/content", "/kaggle/working", os.getcwd()] if os.path.isdir(p))
os.chdir(BASE)
if os.path.isdir("copywrite_repo/.git"):
    !git -C copywrite_repo pull -q
else:
    !git clone -q $REPO_URL copywrite_repo
%cd {BASE}/copywrite_repo
!git log -1 --format="Using commit %h: %s"

In [ ]:
# 3. Install (about 1-2 minutes)
!pip install -q -r requirements.txt -r requirements-model.txt

### 4. Start the app
Starts the app in the background, checks it really runs, and opens a free public link (Cloudflare) that works
on any device. On Colab it also prints a backup Colab link. Re-run this cell to restart the app.

In [ ]:
%run serve.py

### 5. If something doesn't work
Shows the app's log (models loading, every generation, any error) and GPU memory. Copy the output if you need help.

In [ ]:
!python serve.py --status

---
### Optional extras

**Quick model test** - writes two lines in a sample handwriting without the app, checks them with the OCR, and
saves every intermediate image to `debug_test/` (`debug_test/pieces.csv` lists what passed and what was rewritten):

In [ ]:
import time
from copywrite import EmuruGenerator
from copywrite.library import load_library
from copywrite.ocr import get_ocr
from copywrite.writer import LineWriter

ref = load_library()[0]
display(ref.image); print("style text:", ref.text)
gen = EmuruGenerator(); gen.load()
t = time.time()
writer = LineWriter(gen, [ref], ocr=get_ocr(), seed=0, debug_dir="debug_test")
lines = writer.write(["Generative AI mini project for the final year", "handwriting generation with a transformer"])
print(f"{time.time() - t:.1f}s", writer.report.as_dict())
for w in lines.values():
    display(w.image)

**Command line / evaluation** - upload your photo to `samples/style/` first (Files panel on the left):

In [ ]:
# !python -m copywrite.cli --text samples/text/sample_notes.txt --style samples/style/me.jpg --backend emuru --quality best --debug-dir outputs/run1/debug --out outputs/run1
# !pip install -q -r requirements-extra.txt
# !python scripts/evaluate.py --photos samples/style/me_12lines.jpg --text samples/style/me_12lines.txt --k 2 --backends emuru font --out outputs/eval_me